<div style="display: flex; background-color: RGB(255,114,0);" >
<h1 style="margin: auto; padding: 30px; ">Script - Détection de faux billets en euros - Python </h1>
</div>

## Objectif

En tant qu'un consultant sénior Data Analyst. On travaille pour l’Organisation nationale de lutte contre le faux-monnayage (ONCFM) qui cherche à mettre en place des méthodes d’**identification des contrefaçons des billets en euros**. 

Pour se faire, nous allons utiliser les données géométriques collectées par la machine de comptage des billets. Ensuite, nous allons tester 4 méthodes de classification. Une méthode non-supervisée et 3 méthodes supervisée.

**Plan :** 
- Analyse exploratoire des caractéristiques
- Nettoyage des données
- Optimiser les modèles entrainés
- Comparer la performance entre ces modèles


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - Importation des données </h2>
</div>

### Importation des packages

In [3]:
import os
import numpy as np
import scipy.stats as stats
import pandas as pd
import openpyxl
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from mlxtend.feature_selection import SequentialFeatureSelector

from sklearn import metrics
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.metrics import roc_curve
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import silhouette_score

### Importation de données

In [5]:
DATA_PATH = "billets_production.csv"

if not os.path.exists(DATA_PATH):
    print(f"Attention : le fichier '{DATA_PATH}' n'existe pas.")

try:
    billet = pd.read_csv(DATA_PATH, sep=",")
    print("Données chargées :", billet.shape)
    display(billet.head())
    display(billet.info())
except Exception as e:
    print("Erreur lors du chargement :", e)
    print("Tu as bien lu le code avant de l'executer ? ")


Données chargées : (5, 7)


,diagonal,height_left,height_right,margin_low,margin_up,length,id
0,171.76,104.01,103.54,5.21,3.30,111.42,A_1
1,171.87,104.17,104.13,6.00,3.31,112.09,A_2
2,172.00,104.58,104.29,4.99,3.39,111.57,A_3
3,172.49,104.55,104.34,4.44,3.03,113.20,A_4
4,171.65,103.63,103.56,3.77,3.16,113.33,A_5


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   diagonal      5 non-null      float64
 1   height_left   5 non-null      float64
 2   height_right  5 non-null      float64
 3   margin_low    5 non-null      float64
 4   margin_up     5 non-null      float64
 5   length        5 non-null      float64
 6   id            5 non-null      object 
dtypes: float64(6), object(1)
memory usage: 412.0+ bytes


None

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; "> Application de modélisation Logit </h2>
</div>

In [13]:
def script_model_logit(data_path_fit, cols_name, y_name, data_path_pred):
    """ Modèle logit choisit """

    data_fit = pd.read_csv(data_path_fit, sep=",")
    data_pred = pd.read_csv(data_path_pred, sep=",")

    X_fit = data_fit[cols_name]
    y_fit = data_fit[y_name]

    X_pred = data_pred[cols_name]
    
    scaler = StandardScaler()
    data_fit_scaled = scaler.fit_transform(X_fit)
    data_scaled = scaler.fit_transform(X_pred)
    
    # Entrainement
    logit = LogisticRegression(C= 10, 
                               class_weight='balanced', 
                               solver='lbfgs', 
                               max_iter=1000, 
                               random_state = 42)
    logit.fit(data_fit_scaled, y_fit)
    
    # Prediction
    y_pred_prod = logit.predict(data_scaled)

    data_pred["y_pred"] = y_pred_prod
    return data_pred
    


In [14]:
cols = ["diagonal", "height_left",	"height_right", "margin_low", "margin_up", "length"]
data_path = "billet_final.csv"
data_path_pred = "billets_production.csv"
script_model_logit(data_path, cols, "is_genuine", data_path_pred)

,diagonal,height_left,height_right,margin_low,margin_up,length,id,y_pred
0,171.76,104.01,103.54,5.21,3.30,111.42,A_1,1
1,171.87,104.17,104.13,6.00,3.31,112.09,A_2,1
2,172.00,104.58,104.29,4.99,3.39,111.57,A_3,1
3,172.49,104.55,104.34,4.44,3.03,113.20,A_4,0
4,171.65,103.63,103.56,3.77,3.16,113.33,A_5,0


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; "> Application de modélisation Kmeans </h2>
</div>

In [7]:
def script_model_kmeans(cols_name, data_path_pred):
    """ Modèle kmeans choisit """

    data_pred = pd.read_csv(data_path_pred, sep = ",")
    X_pred = data_pred[cols_name]

    # centrée réduite
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(X_pred)

    # 2 axes à prendre en compte
    pca = PCA(n_components=2)
    X_test_pca = pca.fit_transform(data_scaled)

    # kmeans avec 2 clusters
    kmeans = KMeans(n_clusters=2, random_state=42)
    kmeans.fit(X_test_pca)
    
    # Predict
    y_pred_kmeans = kmeans.predict(X_test_pca)

    data_pred["y_pred"] = y_pred_kmeans
    return data_pred
    


In [9]:
cols = ["diagonal", "height_left",	"height_right", "margin_low", "margin_up", "length"]
data_path_pred = "billets_production.csv"
script_model_kmeans(cols, data_path_pred)

,diagonal,height_left,height_right,margin_low,margin_up,length,id,y_pred
0,171.76,104.01,103.54,5.21,3.30,111.42,A_1,0
1,171.87,104.17,104.13,6.00,3.31,112.09,A_2,0
2,172.00,104.58,104.29,4.99,3.39,111.57,A_3,0
3,172.49,104.55,104.34,4.44,3.03,113.20,A_4,0
4,171.65,103.63,103.56,3.77,3.16,113.33,A_5,1


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Fin de notebook </h2>
</div>